## 005_backtest — Leader–Follower Strategy Validation

**Author:** Wayne Kirk Schmidt  
**Email:** wayne.kirk.schmidt@gmail.com  

---

## Stage Purpose

This stage evaluates the statistical arbitrage hypotheses identified
in **004_strategy** by constructing and testing trade rules based on
leader–follower shock propagation.

Using the event-level signal structures produced in 004, we simulate
trades that exploit delayed reactions between assets and measure
the resulting performance characteristics.

The goal of this stage is **not discovery**, but **validation**:
to determine whether the observed signal structures translate into
repeatable, economically meaningful trading performance.

---

## Inputs (from 004_strategy)

Loaded from the pipeline manifest:

- `event_full_df.pkl`
  - Complete event-level dataset of leader–follower relationships
  - Includes:
    - leader coin (`r_coin`)
    - follower coin (`f_coin`)
    - leader direction (`r_sign`)
    - follower directional responses (`f_sign0/1/2`)
    - cumulative responses (`f_sum0/1/2`)

- `event_filtered_df.pkl`
  - Subset of events where follower response aligns directionally
  - Represents candidate signal events

- `event_strong_df.pkl`
  - High-confidence subset where multi-period agreement is observed
  - Represents strongest signal candidates

Additional inputs:

- `returns_full.pkl` (from 002_enrich)
  - Daily return series for all assets

- `price_wide.pkl` (from 002_enrich)
  - Wide-format price series for all assets

These artifacts represent the **frozen output of the strategy construction stage**.

---

## Core Tasks

This notebook performs the following steps:

1. **Signal Definition**
   - Select an input event set (full / filtered / strong)
   - Define trade direction from leader shock (`r_sign`)
   - Define mapping from event structure to executable trade

2. **Trade Construction**
   - Define entry timing (e.g. t+1)
   - Define exit timing (e.g. t+2 or t+3)
   - Map event dates to realized returns using `returns_full`
   - Construct event-level trade records

3. **Trade Simulation**
   - Aggregate event-level trades into a unified trade dataset
   - Apply execution assumptions (delay, cost)
   - Generate trade-level PnL series

4. **Performance Evaluation**
   - cumulative return
   - volatility
   - Sharpe ratio
   - win rate
   - max drawdown

5. **Validation Checks**
   - sensitivity to delay assumptions
   - robustness across event subsets
   - sanity comparison (e.g. randomized baseline)

---

## Outputs (Persisted in `output/005_backtest/`)

Artifacts produced by this stage:

- `trade_df.pkl`
  - Event-level trade log

- `daily_returns.pkl`
  - Aggregated daily return series

- `equity_curve.pkl`
  - Cumulative return curve

- `performance_summary.pkl`
  - Key performance metrics

- `stress_summary.pkl`
  - Performance across cost and delay scenarios

These outputs are registered in the **pipeline manifest** and form the
basis for final evaluation.

---

## Notes

This stage operates exclusively on artifacts generated in
**004_strategy** and **002_enrich**, ensuring that strategy construction
and validation remain cleanly separated within the pipeline.

All results must be reproducible from persisted artifacts.

---

## Design Principles

- **Deterministic pipeline**  
  Signals and trades are generated only from persisted upstream artifacts.

- **Separation of concerns**  
  Strategy construction (004) is distinct from validation (005).

- **Data integrity**  
  Trade construction must use realized returns (`returns_full`), not
  intermediate analysis structures.

- **Transparent mapping**  
  The transformation from event → trade → PnL must be explicit.

- **Artifact persistence**  
  All intermediate and final outputs are saved for inspection and reuse.

---

## Critical Constraint

005 must validate the **full set of signals produced by 004**, not a
manually reduced subset.


### 1. Imports and Environment Setup
### Provide the necessary imports required for to to proceed.   

In [30]:
import pandas as pd
import numpy as np
import pickle
from datetime import datetime, UTC
import math
from pathlib import Path
import matplotlib.pyplot as plt

### 2. Prepare the environment for the notebook

In [31]:
startdate = "2023-01-01"
trading_days = 252
frequency = "1d"

universe = [
    "BTCUSDT",   # Bitcoin
    "ETHUSDT",   # Ethereum
    "BNBUSDT",   # Binance Coin
    "SOLUSDT",   # Solana
    "XRPUSDT",   # Ripple
    "ADAUSDT",   # Cardano
    "DOGEUSDT",  # Dogecoin
    "AVAXUSDT",  # Avalanche
    "LTCUSDT"    # Litecoin
]

execution_delay = [0, 1, 2, 3]
execution_cost_bps = [20, 30, 40]

stage_label = "005_backtest"

OUTPUT_ROOT = Path("../output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = OUTPUT_ROOT / "manifest.pkl"

DOWNLOAD_DIR = OUTPUT_ROOT / "001_download"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

ENRICH_DIR = OUTPUT_ROOT / "002_enrich"
ENRICH_DIR.mkdir(parents=True, exist_ok=True)

ANALYSIS_DIR = OUTPUT_ROOT / "003_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

STRATEGY_DIR = OUTPUT_ROOT / "004_strategy"
STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

BACKTEST_DIR = OUTPUT_ROOT / "005_backtest"
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)

inspection_window = 20

sigma_threshold = 3

observation_window_length = 10
observation_window = range(1, observation_window_length + 1)

holding_period = 1


### 3. Loading the manifest pickle file

In [32]:
if MANIFEST_FILE.exists():
    manifest = pd.read_pickle(MANIFEST_FILE)
else:
    manifest = {}

manifest.setdefault(stage_label, {})

{}

### 4. Load the pickle files from the previous stage
We use these file contents for our analysis.

In [33]:
required_files = {
    "event_response_expanded": STRATEGY_DIR / "event_response_expanded.pkl",
}

missing_files = [
    name for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing required artifacts: {missing_files}")

event_response_expanded = pd.read_pickle(
    required_files["event_response_expanded"]
)

print("Artifacts loaded successfully")

print("\nShape:")
print(event_response_expanded.shape)

print("\nColumns:")
print(event_response_expanded.columns.tolist())

print("\nSample:")
print(event_response_expanded.head())

print("\nNull count:")
print(event_response_expanded.isnull().sum().sum())

Artifacts loaded successfully

Shape:
(2304, 14)

Columns:
['event_date', 'reference_coin', 'target_coin', 'sigma', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_8', 'lag_9', 'lag_10']

Sample:
                 event_date reference_coin target_coin  sigma     lag_1  \
0 2023-08-03 00:00:00+00:00        LTCUSDT     ADAUSDT     -3  0.004098   
1 2023-08-03 00:00:00+00:00        LTCUSDT    AVAXUSDT     -3 -0.003213   
2 2023-08-03 00:00:00+00:00        LTCUSDT     BNBUSDT     -3  0.001663   
3 2023-08-03 00:00:00+00:00        LTCUSDT     BTCUSDT     -3 -0.002569   
4 2023-08-03 00:00:00+00:00        LTCUSDT    DOGEUSDT     -3 -0.001764   

      lag_2     lag_3     lag_4     lag_5     lag_6     lag_7     lag_8  \
0 -0.001020 -0.006129 -0.003768  0.023384  0.011089 -0.014955 -0.009447   
1  0.002417  0.011254 -0.011924  0.021722 -0.004724 -0.011867 -0.004804   
2  0.007884  0.000412 -0.004938  0.014475 -0.006115 -0.010254 -0.005387   
3 -0.001855  0.000888  0.003721  